# GPT-generated summaries of category contents

In [44]:
import pandas as pd
from discovery_child_development import PROJECT_DIR, S3_BUCKET
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [161]:
OUTPUTS_DIR = ENRICHED_DATA_DIR / 'themes'
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)

In [66]:
from discovery_child_development.utils.openai_utils import client

In [4]:
relevant_df = pd.read_csv(ENRICHED_DATA_DIR / 'relevant_labelled_df.csv')

In [6]:
relevant_df.sample(2)

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
7047,KR-20210072202-A,Method for Outputting Infants Emotional Condit...,Patents,"ai2, infancy",0.998998,1,0.069925,0,2021,KR
43524,W4294384666,Validation of the neurocognitive battery PreAc...,Publications,"preschool, cognitive",0.546417,1,0.301184,0,2022,NaN


In [27]:
# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

topics_df = []
for topic in topics_dict:
    topics_df.append({
        'topic': topic,
        'type': topics_dict[topic]['type'],
        'name': topics_dict[topic]['name'],
        # 'description': topics_dict[topic]['description']
    })
topics_df = (
    pd.DataFrame(topics_df)
    .sort_values(['type', 'topic',])
    .reset_index(drop=True)
    .replace('Family and home', 'Parenting')
    .rename(columns={'type': 'Type'})
)

38


In [28]:
topics_df.head()

,topic,Type,name
0,arts,Area of learning,Expressive arts and design
1,communication,Area of learning,Communication and language
2,emotional,Area of learning,Personal social emotional
3,literacy,Area of learning,Literacy
4,mathematics,Area of learning,Mathematics


In [43]:
from discovery_child_development.getters.openalex import get_sentence_embeddings

# Path to sentence embeddings
VECTORS_PATH = "data/outputs/vectors/"
VECTORS_FILE = "sentence_vectors_384_labelled.parquet"

2024-04-23 11:06:03,494 - botocore.credentials - INFO - Found credentials in environment variables.
2024-04-23 11:06:05,093 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
# Load dataset sentence embeddings (all-MiniLM-L6-v2)
embeddings_all = (
    get_sentence_embeddings(
        s3_bucket=S3_BUCKET,
        filepath=VECTORS_PATH,
        filename=VECTORS_FILE,
        id="id",
    )
    .reset_index()
    # Simplify the id by removing https
    .assign(id=lambda df: df["id"].apply(lambda x: x.split("/")[-1]))
    .set_index("id")
)

In [110]:
relevant_df.query("id == 'CN-111870085-A'")

,id,text,Dataset,topics,Detection,Detection_,Management,Management_,year,country_code
4305,CN-111870085-A,An artificial intelligence crib. The invention...,Patents,"infancy, ai2, mathematics, arts, sleep, physical",0.002197,0,0.993387,1,2020,CN


## Pre-process data

In [13]:
def get_overlaps(df: pd.DataFrame, categories: list, category_column: str) -> pd.DataFrame:
    """Fetch the datapoints that have all the categories in the list

    Args:
        df (pd.DataFrame): DataFrame containing the datapoints
        categories (list): List of categories
        category_column (str): Column name that contains the categories

    Returns:
        pd.DataFrame: DataFrame containing the datapoints that have all the categories in the list
    """
    return (
        df
        .copy()
        # transform comma separated string to list, account for nulls
        .assign(**{category_column: lambda x: x[category_column].fillna('').str.split(', ')})
        # filter the rows that have all the categories in the list
        .loc[lambda x: x[category_column].apply(lambda y: set(categories).issubset(y))]
    )

In [135]:
themes = []

In [164]:
types = ["Area of learning", "Development", "Social", "General"]
topic1 = 'mobile'
topics2 = list(set(topics_df.query("Type == @types").topic.to_list()).difference([topic1]))

# for topic2 in topics2[0:1]: 
for topic2 in topics2: 

    for dataset in ["Publications", "Patents"]:

        # Get the documents that have both topics
        df = (
            get_overlaps(relevant_df, [topic1, topic2], 'topics')[['id', 'text', 'topics', 'Dataset']]
            .query("Dataset == @dataset")
            .assign(text_ = lambda x: 'ID: ' + x['id'] + ' | TEXT: ' + x['text'])
        )
        # Take 50 documents which might be most focussed on the two topics
        df = (
            df
            # shuffle
            .sample(len(df))
            # sort by number of topics
            .assign(n_topics = lambda df: df['topics'].apply(lambda x: len(x)))
            .sort_values('n_topics')
            # get the first 50
            .head(50)
        )

        topic1_name = topics_df.query("topic == @topic1").iloc[0]["name"]
        topic2_name = topics_df.query("topic == @topic2").iloc[0]["name"]

        topic1_type = topics_df.query("topic == @topic1").iloc[0]["Type"]
        topic2_type = topics_df.query("topic == @topic2").iloc[0]["Type"]

        abstract_texts = "\n\n".join(df.text_.to_list())

        gpt_message = f"You are an expert researcher on early childhood development. \
            Here are documents related to topics: Topic 1: {topic1_name} ({topic1_type}) and Topic 2: {topic2_name} ({topic2_type}). \
            Detect one to three distinct themes across these documents, that are closely related to the interaction between these two topics, and summarise these themes. \
            For each theme, write up to two sentences of summary, highlighting how the two topics overlap \
            and also two or three most interesting examples of innovations, technologies or interventions, referencing the alphanumeric ID of the document describing the example. \
            Focus on themes and examples that consider both topics together. \
            Here's an example: \n\n##Example\n\nTheme: [Theme name]\nSummary:[Summary of the theme]\nExample 1: \
            [Example of innovation, technology or intervention] (ID: [ID of the document] \nExample 2: [Example of innovation, technology or intervention] \
            (ID: [ID of the document])\n\n##Document texts\n\n {abstract_texts} \n\n##Themes\n\n"
        
        # Generate cluster descriptions
        print(f"Generating cluster themes for {topic1_name} and {topic2_name}")
        print(f"Dataset: {dataset}, number of documents: {len(df)}")
        messages = [
            {
                "role": "user",
                "content": gpt_message,
            }
        ]
        if len(df) > 0:
            chatgpt_output = client.chat.completions.create(
                model="gpt-4-turbo-2024-04-09",
                messages=messages,
                temperature=0.6,
                max_tokens=2000,
            )
            cluster_themes = chatgpt_output.choices[0].message.content
        else:
            print("No documents found for the given topics") 

        themes.append(
            {
                "topic1": topic1,
                "topic2": topic2,
                "dataset": dataset,
                "numb_docs": len(df),
                "cluster_themes": cluster_themes,
            }
        )
    

Generating cluster themes for Mobile and Prenatal
Dataset: Publications, number of documents: 14
2024-04-23 15:58:37,646 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Prenatal
Dataset: Patents, number of documents: 10
2024-04-23 15:58:51,082 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Labour market
Dataset: Publications, number of documents: 9
2024-04-23 15:59:09,001 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generating cluster themes for Mobile and Labour market
Dataset: Patents, number of documents: 0
No documents found for the given topics
Generating cluster themes for Mobile and Community
Dataset: Publications, number of documents: 30
2024-04-23 15:59:39,297 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 

In [169]:
df

,id,text,topics,Dataset,text_,n_topics
79,KR-101708776-B1,A cooperative child-care service system using ...,"[mobile, infancy]",Patents,ID: KR-101708776-B1 | TEXT: A cooperative chil...,2
6004,CN-211184123-U,a baby surveillance camera. The utility model ...,"[mobile, infancy]",Patents,ID: CN-211184123-U | TEXT: a baby surveillance...,2
8835,CN-206726755-U,A kind of electronics windbell of band infant ...,"[mobile, infancy]",Patents,ID: CN-206726755-U | TEXT: A kind of electroni...,2
6679,CN-107610410-A,A kind of child missing alarm. The present inv...,"[mobile, infancy]",Patents,ID: CN-107610410-A | TEXT: A kind of child mis...,2
1231,KR-20190129169-A,Device for monitoring baby&#39;s condition and...,"[mobile, infancy]",Patents,ID: KR-20190129169-A | TEXT: Device for monito...,2
3592,CN-209692931-U,The bright stove machine in bright kitchen of ...,"[mobile, infancy]",Patents,ID: CN-209692931-U | TEXT: The bright stove ma...,2
10170,CN-111166193-A,A mobile constant temperature milk storage dev...,"[mobile, infancy]",Patents,ID: CN-111166193-A | TEXT: A mobile constant t...,2
11406,WO-2015017367-A1,Bidirectional communication between an infant ...,"[mobile, infancy]",Patents,ID: WO-2015017367-A1 | TEXT: Bidirectional com...,2
10950,KR-20180129569-A,Indirect experience of Education Application b...,"[mobile, infancy]",Patents,ID: KR-20180129569-A | TEXT: Indirect experien...,2
1641,KR-200479571-Y1,Cellpfone case with tooth development period. ...,"[mobile, infancy]",Patents,ID: KR-200479571-Y1 | TEXT: Cellpfone case wit...,2


In [165]:
themes_df = pd.DataFrame(themes).rename(columns={'numb_docs': 'n_docs'})
# if numb_docs = 0, then make cluster_themes empty
themes_df.loc[themes_df['n_docs'] == 0, 'cluster_themes'] = ''

In [166]:
topic1

'mobile'

In [167]:
# Save json and save csv
themes_df.to_csv(OUTPUTS_DIR / f'topic_themes_{topic1}.csv', index=False)
themes_df.to_json(OUTPUTS_DIR / f'topic_themes_{topic1}.json', orient='records')

In [171]:
topics_df.to_csv(OUTPUTS_DIR / 'topics.csv', index=False)

In [118]:


# # Get cluster centroid indices
# centroids = get_cluster_centroids(cluster_df, embeddings)
# most_central = []
# for i in range(len(centroids)):
#     most_central.append(
#         get_n_most_central_vectors(embeddings, centroids[i], n=n_central)
#     )



Generating cluster themes for Data science and AI and Mathematics
Dataset: Publications, number of documents: 29
2024-04-23 12:24:09,124 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Summarisation